In [ ]:
%pip install torch numpy scikit-learn matplotlib google-cloud-storage google-auth

In [ ]:
import os
import sys

REPO_URL = "https://github.com/payamdav/pycrypto.git"
REPO_NAME = "pycrypto"
BRANCH_NAME = "claude/gallant-clarke-sN18M"

if not os.path.exists(REPO_NAME):
    !git clone -b {BRANCH_NAME} {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import mean_squared_error, mean_absolute_error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
ASSET = "btcusdt"
# ASSET = "ethusdt"
# ASSET = "xrpusdt"
# ASSET = "adausdt"
# ASSET = "dogeusdt"
# ASSET = "trumpusdt"
# ASSET = "vineusdt"

VWAP_IDX = 2

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
PURGE_GAP   = 1440 * 7   # minimum index distance between splits (7 days of 1-min candles)

BATCH_SIZE      = 2048
EPOCHS          = 50
LEARNING_RATE   = 1e-3
WEIGHT_DECAY    = 1e-4

print(f"Asset: {ASSET.upper()}")
print(f"VWAP_IDX: {VWAP_IDX}")
print(f"Train/Val/Test: {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")
print(f"Purge gap: {PURGE_GAP:,} rows ({PURGE_GAP // 1440} days)")
print(f"Batch size: {BATCH_SIZE}, Epochs: {EPOCHS}, LR: {LEARNING_RATE}, WD: {WEIGHT_DECAY}")

In [ ]:
import io

sys.path.insert(0, "pycrypto/packages/tools/google_cloud_storage_tools")
from gcs_tools import gcs_json_key_file, read_file

key_path = gcs_json_key_file()  # MUST be called first
print(f"GCS credentials resolved: {key_path}")

BUCKET = "payamdprojectbucket"
print(f"Loading fl_data_{ASSET} from gs://{BUCKET}/...")

raw_bytes = read_file(BUCKET, f"fl_data_{ASSET}")
data = np.load(io.BytesIO(raw_bytes))

print(f"Data shape: {data.shape}")
print(f"Total rows: {len(data):,}")

In [ ]:
# Features: f_e_vwap only (cols 1-24)
# Shape: (N, 24, 1)
X = data[:, 1:25].reshape(-1, 24, 1).astype(np.float32)

# Labels: single vwap head at index VWAP_IDX from l_e_vwap (cols 52-55)
# Shape: (N, 1)
y = data[:, 52 + VWAP_IDX].reshape(-1, 1).astype(np.float32)

print("X shape:", X.shape)  # (N, 24, 1)
print("y shape:", y.shape)  # (N, 1)
print(f"X min: {X.min():.4f}, X max: {X.max():.4f}")
print(f"y min: {y.min():.4f}, y max: {y.max():.4f}")

In [ ]:
N = len(X)
train_end   = int(N * TRAIN_RATIO)
val_start   = train_end + PURGE_GAP
val_end     = val_start + int(N * VAL_RATIO)
test_start  = val_end + PURGE_GAP
test_end    = test_start + int(N * TEST_RATIO)

X_train, y_train = X[:train_end], y[:train_end]
X_val,   y_val   = X[val_start:val_end], y[val_start:val_end]
X_test,  y_test  = X[test_start:test_end], y[test_start:test_end]

print(f"Total rows  : {N:,}")
print(f"Train       : {len(X_train):,}  (rows 0 — {train_end-1:,})")
print(f"Purge gap 1 : {PURGE_GAP:,}  (rows {train_end:,} — {val_start-1:,})")
print(f"Val         : {len(X_val):,}  (rows {val_start:,} — {val_end-1:,})")
print(f"Purge gap 2 : {PURGE_GAP:,}  (rows {val_end:,} — {test_start-1:,})")
print(f"Test        : {len(X_test):,}  (rows {test_start:,} — {test_end-1:,})")

In [ ]:
def make_loader(X, y, batch_size, shuffle=False):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, pin_memory=True, num_workers=2)

train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   BATCH_SIZE)
test_loader  = make_loader(X_test,  y_test,  BATCH_SIZE)

print(f"Train batches: {len(train_loader):,}")
print(f"Val   batches: {len(val_loader):,}")
print(f"Test  batches: {len(test_loader):,}")

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, dropout=0.3, output_size=1):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.drop2 = nn.Dropout(dropout)
        self.fc    = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm1(x)              # (B, T, hidden)
        out     = self.drop1(out)
        out, _ = self.lstm2(out)            # (B, T, hidden)
        out     = self.drop2(out[:, -1, :]) # take final hidden state
        return torch.tanh(self.fc(out))     # (B, 1)


model = LSTMRegressor(input_size=1, output_size=1).to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
criterion = nn.HuberLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

print(f"Loss     : HuberLoss")
print(f"Optimizer: AdamW  lr={LEARNING_RATE}  wd={WEIGHT_DECAY}")
print(f"Scheduler: ReduceLROnPlateau  factor=0.5  patience=3")

In [ ]:
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # --- Train ---
    model.train()
    train_loss = 0.0
    n_batches = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        n_batches += 1
    train_loss /= n_batches

    # --- Validate ---
    model.eval()
    val_loss = 0.0
    n_val_batches = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_loss += criterion(model(xb), yb).item()
            n_val_batches += 1
    val_loss /= n_val_batches

    scheduler.step(val_loss)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    elapsed = time.time() - t0
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:3d}/{EPOCHS} | {elapsed:.1f}s | batches={n_batches} | "
          f"train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | lr={current_lr:.2e}")

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(epochs_range, history["train_loss"], label="Train Loss", linewidth=2)
ax.plot(epochs_range, history["val_loss"],   label="Val Loss",   linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Huber Loss")
ax.set_title(f"Training Curves — {ASSET.upper()}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
model.eval()
test_preds_list  = []
test_truths_list = []
test_huber_loss  = 0.0
n_test_batches   = 0

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        test_huber_loss += criterion(pred, yb).item()
        n_test_batches += 1
        test_preds_list.append(pred.cpu().numpy())
        test_truths_list.append(yb.cpu().numpy())

test_huber_loss /= n_test_batches
test_preds  = np.concatenate(test_preds_list,  axis=0)
test_truths = np.concatenate(test_truths_list, axis=0)

test_mse = mean_squared_error(test_truths, test_preds)
test_mae = mean_absolute_error(test_truths, test_preds)

print(f"Test Huber Loss : {test_huber_loss:.6f}")
print(f"Test MSE        : {test_mse:.6f}")
print(f"Test MAE        : {test_mae:.6f}")
print()

head_name = f"l_e_vwap_{VWAP_IDX + 1}"
print(f"{'Head':<18} {'MSE':>10} {'MAE':>10}")
print("-" * 42)
h_mse = mean_squared_error(test_truths[:, 0], test_preds[:, 0])
h_mae = mean_absolute_error(test_truths[:, 0], test_preds[:, 0])
print(f"{head_name:<18} {h_mse:>10.6f} {h_mae:>10.6f}")

In [ ]:
# import io
# import sys
# sys.path.insert(0, "pycrypto/packages/tools/google_cloud_storage_tools")
# from gcs_tools import gcs_json_key_file, write_file
# key_path = gcs_json_key_file()
# BUCKET = "payamdprojectbucket"
# buffer = io.BytesIO()
# torch.save(model.state_dict(), buffer)
# buffer.seek(0)
# write_file(BUCKET, f"lstm_model_{ASSET}.pt", buffer, content_type="application/octet-stream")
# print(f"Model saved to gs://{BUCKET}/lstm_model_{ASSET}.pt")

In [ ]:
# Run full inference on val set
all_preds  = []
all_truths = []
model.eval()
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        all_preds.append(model(xb).cpu().numpy())
        all_truths.append(yb.numpy())

preds  = np.concatenate(all_preds,  axis=0)  # (N_val, 1)
truths = np.concatenate(all_truths, axis=0)  # (N_val, 1)

print(f"Val inference complete: preds {preds.shape}, truths {truths.shape}")

In [ ]:
thresholds = np.arange(0.50, 0.96, 0.01)
head_name = f"l_e_vwap_{VWAP_IDX + 1}"

fig, ax = plt.subplots(figsize=(8, 5))
precisions = []
counts     = []
for tau in thresholds:
    predicted_pos = preds[:, 0] >= tau
    actual_pos    = truths[:, 0] >= tau
    tp        = (predicted_pos & actual_pos).sum()
    pp        = predicted_pos.sum()
    precision = tp / pp if pp > 0 else 0.0
    precisions.append(precision)
    counts.append(pp)

ax.plot(thresholds, precisions, color="steelblue", linewidth=2, label="Precision")
ax2 = ax.twinx()
ax2.plot(thresholds, counts, color="orange", alpha=0.5, linewidth=1.5, label="Signal count")
ax.set_title(head_name)
ax.set_xlabel("Threshold \u03c4")
ax.set_ylabel("Precision")
ax2.set_ylabel("# Signals")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)

plt.suptitle(f"Precision vs Threshold — {head_name} — {ASSET.upper()}", fontsize=14)
plt.tight_layout()
plt.show()